In [1]:
import  pandas as pd

In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np

class CrudeDataset(Dataset):

    mapping = {f"ch{i}": i for i in range(8)}
    feature_columns = ['PRN', 'Carrier_Doppler_hz', 'Pseudorange_m', 'RX_time', 
                        'TOW', 'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 
                        'TCD', 'CN0']

    def __init__(self, df, target_time_series, seq_len=15):
        self.df = df.copy()  # Make a copy to avoid modifying original

        # Ensure correct types early - keep as int for sorting
        self.df["channel"] = self.df["channel"].astype(int)

        # Get unique times for indexing - ensure they're sorted
        self.target_time_series = sorted(target_time_series)
        self.seq_len = seq_len

    def __len__(self):
        # Only return indices that have enough history (seq_len previous timesteps)
        return len(self.target_time_series) - self.seq_len

    def __getitem__(self, idx):
        # idx is offset by seq_len since we need seq_len previous timesteps
        actual_idx = idx +self.seq_len
        
        # Get target time
        target_time = self.target_time_series[actual_idx]
                
        # Get target from the target timestep
        target_data = self.df[self.df["time"] == target_time]
        target_label = target_data["spoofed"].values.mean()
        target_features = self.flatten_single_timestep(target_data)
        
        # Collect features for all timesteps in the window
        features = []
        for t in range(target_time-self.seq_len,target_time):
            time_data = self.df[self.df["time"] == t]
            single_timestep_features = self.flatten_single_timestep(time_data)
            features.append(single_timestep_features)
        
        # Convert to tensors
        features = torch.from_numpy(np.array(features)).float()
        target_features = torch.from_numpy(target_features).float()
        target_label = torch.tensor(target_label).float()
        target_time_tensor = torch.tensor(target_time).float()
        
        # Handle numerical issues
        features = torch.nan_to_num(features, nan=0.0, posinf=10.0, neginf=-10.0)
        target_features = torch.nan_to_num(target_features, nan=0.0, posinf=10.0, neginf=-10.0)
        
        # Clamp to stable range
        features = torch.clamp(features, -5.0, 5.0)
        target_features = torch.clamp(target_features, -5.0, 5.0)
        
        return target_label, features, target_features, target_time_tensor
    
    def flatten_single_timestep(self, time_data):
        """
        Flatten a single timestep's data (8 channels × 14 features = 112 features)
        """
        # Ensure we have all 8 channels
        if len(time_data) != 8:
            # If missing channels, create a complete dataframe with zeros
            complete_data = []
            for ch in range(8):
                ch_data = time_data[time_data["channel"] == ch]
                if len(ch_data) == 0:
                    # Create zero row for missing channel
                    zero_row = {col: 0 for col in CrudeDataset.feature_columns}
                    zero_row["channel"] = ch
                    complete_data.append(zero_row)
                else:
                    complete_data.append(ch_data.iloc[0].to_dict())
            time_data = pd.DataFrame(complete_data)
        
        # Sort by channel to ensure consistent order
        time_data_sorted = time_data.sort_values('channel')
        
        # Flatten all features for all 8 channels
        flattened = time_data_sorted[CrudeDataset.feature_columns].values.flatten()
        
        return flattened  # Shape: (112,)

In [3]:
df = pd.read_csv("../dataset/test.csv")

In [4]:
df

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0
0,111402,ch0,8,4749.068417,4.841489e+06,263154.82,263154.803851,-435396.111594,101998.429688,100788.812500,111415.289062,-107731.039062,-28414.615234,4660.467773,43.559540
1,111402,ch1,9,1995.777378,2.449848e+06,263154.82,263154.811828,-201479.950180,100812.039062,98424.367188,109174.476562,-105347.234375,-28653.570312,1997.164185,47.116425
2,111402,ch2,27,3458.024120,3.738822e+06,263154.82,263154.807529,-356000.635312,138335.140625,125640.570312,138446.281250,137254.781250,-18124.476562,3467.803955,45.687332
3,111402,ch3,26,-2105.992470,5.482371e+06,263154.82,263154.801713,193102.504674,84470.195312,84153.156250,95917.468750,95899.140625,1874.903442,-2109.972656,42.264256
4,111402,ch4,4,-493.638193,2.666526e+06,263154.82,263154.811105,48203.346004,103226.210938,100365.203125,114574.656250,-114160.539062,-9732.571289,-494.215881,43.629795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381947,159145,ch3,0,0.000000,0.000000e+00,264109.68,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
381948,159145,ch4,0,0.000000,0.000000e+00,264109.68,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
381949,159145,ch5,9,1206.630594,1.298429e+06,264109.68,264109.675669,-136131.389426,141574.875000,139169.453125,156087.781250,156078.093750,-1738.840820,2778.508789,47.498169
381950,159145,ch6,27,2815.545864,2.298911e+06,264109.68,264109.672332,-301032.021016,164279.406250,149921.359375,177528.125000,175577.578125,26244.027344,2786.775146,46.232616


In [5]:
df_train = pd.read_csv("../dataset/train.csv")

/tmp/ipykernel_23420/1750772546.py:1: DtypeWarning: Columns (0: PRN, 1: Carrier_Doppler_hz, 2: Pseudorange_m, 3: RX_time, 4: TOW, 5: Carrier_phase, 6: EC, 7: LC, 8: PC, 9: PIP, 10: PQP, 11: TCD, 12: CN0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv("../dataset/train.csv")


In [6]:
cols=[]
for col in df_train.columns:
    if df_train[col].dtype == "object":
        print(col)
        cols.append(col)

PRN
Carrier_Doppler_hz
Pseudorange_m
RX_time
TOW
Carrier_phase
EC
LC
PC
PIP
PQP
TCD
CN0


In [7]:
df_train[cols] = df_train[cols].apply(pd.to_numeric, errors='coerce')

In [9]:
df_train.fillna(0,inplace=True)

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
0,0,ch0,0.0,0.000000,0.000000e+00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,0,ch1,0.0,0.000000,0.000000e+00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
2,0,ch2,0.0,0.000000,0.000000e+00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
3,0,ch3,0.0,0.000000,0.000000e+00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
4,0,ch4,0.0,0.000000,0.000000e+00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891211,111401,ch3,26.0,-2113.491414,5.482363e+06,263154.8,263154.781713,193060.427210,101265.445312,96612.585938,104833.250000,-104328.625000,-10273.680664,-2108.602539,42.208794,0
891212,111401,ch4,4.0,-493.836685,2.666525e+06,263154.8,263154.791105,48193.433748,112036.953125,122600.250000,124538.890625,124537.945312,-485.292389,-495.879303,43.671265,0
891213,111401,ch5,0.0,0.000000,0.000000e+00,263154.8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
891214,111401,ch6,0.0,0.000000,0.000000e+00,263154.8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0


In [10]:
111401-15

111386

In [11]:
df_train = df_train[df_train["time"]>111386]

In [12]:
df_train.shape

(120, 16)

In [13]:
df_new = pd.concat([df_train,df])

In [14]:
df_new

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
891096,111387,ch0,8.0,4754.469613,4.841762e+06,263154.52,263154.503850,-433970.696997,74156.445312,103278.828125,105496.515625,103500.843750,-20422.798828,4651.393066,43.145805,0.0
891097,111387,ch1,9.0,1998.833763,2.449960e+06,263154.52,263154.511828,-200880.879614,105618.906250,122251.460938,126154.195312,-125825.835938,9096.207031,1992.035522,46.730450,0.0
891098,111387,ch2,27.0,3456.676635,3.739020e+06,263154.52,263154.507528,-354962.733358,123909.945312,131904.640625,143434.031250,-138459.890625,37445.722656,3464.111084,45.249653,0.0
891099,111387,ch3,26.0,-2108.371910,5.482251e+06,263154.52,263154.501713,192471.785222,77786.984375,78865.343750,93150.515625,-92208.570312,-13213.559570,-2108.321777,41.278675,0.0
891100,111387,ch4,4.0,-495.149225,2.666498e+06,263154.52,263154.511106,48054.653385,99923.710938,108026.265625,126252.921875,126178.609375,-4331.131836,-495.663544,44.090580,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
381947,159145,ch3,0.0,0.000000,0.000000e+00,264109.68,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
381948,159145,ch4,0.0,0.000000,0.000000e+00,264109.68,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
381949,159145,ch5,9.0,1206.630594,1.298429e+06,264109.68,264109.675669,-136131.389426,141574.875000,139169.453125,156087.781250,156078.093750,-1738.840820,2778.508789,47.498169,NaN
381950,159145,ch6,27.0,2815.545864,2.298911e+06,264109.68,264109.672332,-301032.021016,164279.406250,149921.359375,177528.125000,175577.578125,26244.027344,2786.775146,46.232616,NaN


In [21]:
target_time = df_new.time.unique()

In [22]:
target_time

array([111387, 111388, 111389, ..., 159143, 159144, 159145],
      shape=(47759,))

In [23]:
target_time[0+15]

np.int64(111402)

In [19]:
df.time.max()

np.int64(159145)

In [25]:
df_new["channel"] = df_new["channel"].map(CrudeDataset.mapping)

In [26]:
testing_dataset = CrudeDataset(df_new,target_time_series=target_time)

In [28]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [49]:
test_loader = DataLoader(testing_dataset, batch_size=32, shuffle=False)


In [50]:
_, features, transformed_target,time = next(iter(test_loader))
print("Target shape:", time.shape)  # (batch_size, 1)
print("Features shape:", features.shape)  # (batch_size, seq_len, num_features)
print("Transformed Target shape:", transformed_target.shape)  # (batch_size, num_features)

Target shape: torch.Size([32])
Features shape: torch.Size([32, 15, 104])
Transformed Target shape: torch.Size([32, 104])


In [39]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim=14, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=256, dropout=0.1,):
        super(TimeSeriesTransformer, self).__init__()
        
        self.d_model = d_model
                
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
                
        # Output layers
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Prediction head
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, input_dim)
        )
        
    def forward(self, x):
        batch_size, seq_len, input_dim = x.shape
        # Reshape for transformer
        x = x.view(batch_size , seq_len, input_dim)
        
        # Project input
        x = self.input_projection(x) * math.sqrt(self.d_model)
        # Add positional encoding
        x = self.pos_encoder(x)
        
        residual = x[:, -1, :]  # (batch, input_dim)

        # Apply transformer
        x = self.transformer(x)  # (batch, seq_len, d_model)
        
        # Take last timestep
        x = x[:, -1, :]  # (batch, d_model)
        
        # Reshape back
        x = x.view(batch_size, self.d_model)
        
        # Apply layer norm and residual
        x = self.layer_norm1(x+residual)
        x = self.dropout(x)

        feature = x
        
        # Output projection
        output = self.output_projection(x)
        
        return output, feature

In [40]:
class Head(nn.Module):
    def __init__(
        self,
        dim: int = 18,
        hidden_dim: int = 64,
        dropout: float = 0.1
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),

            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)  # (B, 1) → RAW LOGITS

In [41]:
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [44]:
timeseries_model=TimeSeriesTransformer(input_dim=features.shape[2]).to(device)

In [43]:
head_model = Head(dim=336, hidden_dim=320).to(device)

In [45]:
timeseries_model.load_state_dict(torch.load('new_finetuned_with_head.pth')['timeseries_model'])
head_model.load_state_dict(torch.load('new_finetuned_with_head.pth')['head_model'])

<All keys matched successfully>

In [46]:
target_time.shape

(47759,)

In [48]:
(47759-15)/32

1492.0

In [52]:
timeseries_model.load_state_dict(torch.load('new_finetuned_with_head.pth')['timeseries_model'])
head_model.load_state_dict(torch.load('new_finetuned_with_head.pth')['head_model'])

y_pred_list=[]
times=[]
i=0

for _, features, transformed_target, time in test_loader:
    features = features.to(device)
    time = time.to(device)
    transformed_target = transformed_target.to(device)

    output,temp = timeseries_model(features)
    temp = torch.tensor(temp, dtype=torch.float32).to(device)
    fusion = torch.cat([output, transformed_target, temp], dim=-1)

    y_pred = head_model(fusion)

    y_pred_list.append(y_pred.detach().cpu().numpy())
    times.append(time.cpu().numpy().reshape(-1))

    if i%10==0:
        print(i)
    i+=1


/tmp/ipykernel_23420/1708470994.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  temp = torch.tensor(temp, dtype=torch.float32).to(device)


0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
1450
1460
1470
1480
1490


In [53]:
# Concatenate predictions
y_pred_prob = np.concatenate(y_pred_list, axis=0)
times = np.concatenate(times, axis=0)

times = times.reshape(-1)

# Ensure 1D arrays
y_pred_prob = y_pred_prob.reshape(-1)
# Apply sigmoid to logits
y_pred_prob = 1 / (1 + np.exp(-y_pred_prob))

# Convert to binary labels
y_pred_label = (y_pred_prob > 0.5).astype(int)


In [54]:
df_res = pd.DataFrame(
    {
        "time": times,
        "spoofed": y_pred_label,
        "confidence":y_pred_prob
    }
)

df_res.to_csv("../new_result.csv",index=False)

In [56]:
df_res.spoofed.value_counts()

spoofed
0    43715
1     4029
Name: count, dtype: int64

In [58]:
df.time.unique().shape

(47744,)